# 課題② PyTorch で 2層NN による MNIST 分類

**`# TODO:` のセルを自分で完成させてください。**  
わからない場合は `solutions/sol_02_mnist_nn.ipynb` を参照してください。

---

## セットアップ

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'使用デバイス: {device}')
torch.manual_seed(42)

---

## Step 1: データの読み込み（完成済み）

In [ ]:
data_root = '/data/shared/datasets' if os.path.exists('/data/shared') else './data'
print(f'データ保存先: {data_root}')

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root=data_root, train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root=data_root, train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f'訓練データ: {len(train_dataset)} 枚 / テストデータ: {len(test_dataset)} 枚')

---

## Step 2: モデルの定義

以下の構造を持つ 2層NN を `nn.Module` で実装してください。

```
入力: (batch, 784)
  ↓  fc1: Linear(784 → 256) + ReLU
  ↓  fc2: Linear(256 → 10)
出力: (batch, 10)  ← 各クラスのスコア（logit）
```

In [ ]:
class TwoLayerNN(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, output_dim=10):
        super().__init__()
        # TODO: fc1（Linear: input_dim → hidden_dim）を定義してください
        self.fc1  = None
        self.relu = nn.ReLU()
        # TODO: fc2（Linear: hidden_dim → output_dim）を定義してください
        self.fc2  = None

    def forward(self, x):
        # TODO: x を (batch, 784) に展開してください（x.view を使用）
        x = None
        # TODO: fc1 → ReLU を通してください
        x = None
        # TODO: fc2 を通して返してください
        return None

# 確認
model = TwoLayerNN().to(device)
dummy = torch.zeros(4, 1, 28, 28).to(device)
out = model(dummy)
assert out.shape == (4, 10), f'出力形状が正しくありません: {out.shape}'
print('TwoLayerNN ✓')
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\n総パラメータ数: {total_params:,}')

---

## Step 3: 損失関数と最適化アルゴリズム（完成済み）

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print('損失関数: CrossEntropyLoss')
print('最適化:   Adam (lr=0.001)')

---

## Step 4: 学習ループの実装

PyTorch 学習ループの **5ステップ** を実装してください。

1. **順伝播**: `model(imgs)` で予測値を計算
2. **損失計算**: `criterion(outputs, labels)` で損失を計算
3. **勾配リセット**: `optimizer.zero_grad()` で前ステップの勾配をクリア
4. **逆伝播**: `loss.backward()` で勾配を計算
5. **パラメータ更新**: `optimizer.step()` で重みを更新

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """1エポック分の訓練を行い、平均 loss と accuracy を返す"""
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        # TODO: 5ステップを順番に実装してください
        outputs = None     # 1. 順伝播
        loss    = None     # 2. 損失計算
        # 3. 勾配リセット
        # 4. 逆伝播
        # 5. パラメータ更新

        total_loss += loss.item() * imgs.size(0)
        correct    += (outputs.argmax(dim=1) == labels).sum().item()
        total      += imgs.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    """テストデータで評価し、平均 loss と accuracy を返す"""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    # TODO: torch.no_grad() コンテキスト内でループを実装してください
    # ヒント: train_epoch と同様の集計ロジック（ただし backward/step は不要）
    pass

    return total_loss / total, correct / total

In [ ]:
# 実行時間: CPU で約3〜5分、GPU で約30秒

n_epochs = 5
train_losses, train_accs = [], []
test_losses,  test_accs  = [], []

for epoch in range(1, n_epochs + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    te_loss, te_acc = evaluate(model, test_loader, criterion, device)

    train_losses.append(tr_loss); train_accs.append(tr_acc)
    test_losses.append(te_loss);  test_accs.append(te_acc)

    print(f'Epoch {epoch}/{n_epochs}  '
          f'train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  '
          f'test_loss={te_loss:.4f}  test_acc={te_acc:.4f}')

---

## Step 5: 学習曲線の可視化（完成済み）

In [ ]:
epochs = range(1, n_epochs + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(epochs, train_losses, color='#0D4A38', linewidth=2, marker='o', markersize=5, label='Train')
ax1.plot(epochs, test_losses,  color='#7C5C00', linewidth=2, marker='s', markersize=5, label='Test', linestyle='--')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('損失曲線')
ax1.legend(); ax1.set_xticks(epochs)

ax2.plot(epochs, [a * 100 for a in train_accs], color='#0D4A38', linewidth=2, marker='o', markersize=5, label='Train')
ax2.plot(epochs, [a * 100 for a in test_accs],  color='#7C5C00', linewidth=2, marker='s', markersize=5, label='Test', linestyle='--')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.set_title('精度曲線')
ax2.legend(); ax2.set_ylim(80, 100); ax2.set_xticks(epochs)

plt.tight_layout()
plt.show()
print(f'最終テスト精度: {test_accs[-1]*100:.2f}%')

---

## チャレンジ問題

`hidden_dim` を `64`、`256`、`512` に変えて、3エポックずつ学習し、  
テスト精度の曲線を1つのグラフに重ねて描いてみましょう。

In [ ]:
# TODO: hidden_dims = [64, 256, 512] で学習を実行し、テスト精度を比較してください
